In [1]:
sys.path.append('/group/pmc021/amunif/env/pytorch/lib/python311.zip')
sys.path.append('/group/pmc021/amunif/env/pytorch/lib/python3.11')
sys.path.append('/group/pmc021/amunif/env/pytorch/lib/python3.11/lib-dynload')
sys.path.append('/home/amunif/.local/lib/python3.11/site-packages')
sys.path.append('/group/pmc021/amunif/env/pytorch/lib/python3.11/site-packages')

In [2]:
import os
import polars as pl
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import ndcg_score, dcg_score, classification_report

import xgboost as xgb

In [3]:
DATASET_DIR = '/group/pmc021/amunif/epi-thesis/workflow/08_HepG2/'
WORKING_DIR = '/group/pmc021/amunif/epi-thesis/workflow/09_Learning to Rank/'

In [4]:
def load_data(path):
    gene_pl = pd.read_parquet(path)
    return gene_pl

In [5]:
def reformat_data(df, columns):
    processed_arrays = []

    for col in columns:
        stacked = np.vstack(df[col].values)
        processed_arrays.append(stacked)

    X = np.hstack(processed_arrays)
    return X

# Load Dataset

In [8]:
# Load dataset
gene_pl = load_data(os.path.join(DATASET_DIR, 'dataset', 'gene_w_qlabel.parquet'))
gene_pl.head(5)

,gene_id,H3K4me3,H3K4me3_count,H3K9ac,H3K9ac_count,H3K9me3,H3K9me3_count,H3K27ac,H3K27ac_count,H3K27me3,H3K27me3_count,value_1,label
0,XLOC_000001,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,0.000000,0
1,XLOC_000003,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,0.000000,0
2,XLOC_000006,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,0.088845,1
3,XLOC_000007,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,4.047430,2
4,XLOC_000008,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",6,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",2,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,26.793400,3


In [13]:
gene_pl.groupby(["label"]).count()

,gene_id,H3K4me3,H3K4me3_count,H3K9ac,H3K9ac_count,H3K9me3,H3K9me3_count,H3K27ac,H3K27ac_count,H3K27me3,H3K27me3_count,value_1
label,,,,,,,,,,,,
0,5539,5539,5539,5539,5539,5539,5539,5539,5539,5539,5539,5539
1,5538,5538,5538,5538,5538,5538,5538,5538,5538,5538,5538,5538
2,5538,5538,5538,5538,5538,5538,5538,5538,5538,5538,5538,5538
3,5539,5539,5539,5539,5539,5539,5539,5539,5539,5539,5539,5539


In [14]:
markers = ['H3K4me3', 'H3K9ac', 'H3K9me3', 'H3K27ac', 'H3K27me3']
X = reformat_data(gene_pl, markers)

In [15]:
X

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [16]:
y = gene_pl['label'].values

In [17]:
gene_ids = gene_pl['gene_id'].values
gene_ids

array(['XLOC_000001', 'XLOC_000003', 'XLOC_000006', ..., 'XLOC_030014',
       'XLOC_030017', 'XLOC_030018'], dtype=object)

In [18]:
value_1 = gene_pl['value_1'].values

In [19]:
y[:20]

array([0, 0, 1, 2, 3, 3, 0, 2, 1, 1, 2, 2, 1, 0, 2, 2, 3, 2, 2, 3])

In [20]:
seed = 1994
rng = np.random.default_rng(seed)
n_query_groups = 1 # Single query group
qid = rng.integers(0, n_query_groups, size=X.shape[0])

# Modeling

In [21]:
ranker = xgb.XGBRanker(
            tree_method="hist", 
            lambdarank_num_pair_per_sample=8, 
            objective="rank:pairwise", 
            lambdarank_pair_method="topk"
        )

In [22]:
X.shape

(22154, 20000)

In [23]:
# Create dataframe for ranking
df = pd.DataFrame(X, columns=[str(i) for i in range(X.shape[1])])
df["qid"] = qid

In [24]:
df

,0,1,2,3,4,5,6,7,8,9,...,19991,19992,19993,19994,19995,19996,19997,19998,19999,qid
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22149,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
22150,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
22151,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
22152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [25]:
ranker.fit(df, y)

XGBRanker(base_score=None, booster=None, callbacks=None, colsample_bylevel=None,
          colsample_bynode=None, colsample_bytree=None, device=None,
          early_stopping_rounds=None, enable_categorical=False,
          eval_metric=None, feature_types=None, gamma=None, grow_policy=None,
          importance_type=None, interaction_constraints=None,
          lambdarank_num_pair_per_sample=8, lambdarank_pair_method='topk',
          learning_rate=None, max_bin=None, max_cat_threshold=None,
          max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
          max_leaves=None, min_child_weight=None, missing=nan,
          monotone_constraints=None, multi_strategy=None, n_estimators=None,
          n_jobs=None, ...)

In [26]:
scores = ranker.predict(X)

In [27]:
df_full = df

In [28]:
df_full['gene_id'] = gene_ids

In [29]:
df_full['value_1'] = value_1

In [30]:
df_full['y'] = y

In [31]:
df_full['scores'] = scores

In [32]:
df_full

,0,1,2,3,4,5,6,7,8,9,...,19995,19996,19997,19998,19999,qid,gene_id,value_1,y,scores
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_000001,0.000000,0,-6.979959
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_000003,0.000000,0,-6.979959
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_000006,0.088845,1,-6.979959
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_000007,4.047430,2,-6.979959
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_000008,26.793400,3,-5.144913
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22149,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_030009,0.000000,0,-6.979959
22150,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_030012,0.000000,0,-6.979959
22151,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_030014,0.000000,0,-6.979959
22152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_030017,0.000000,0,-6.979959


In [33]:
df_full.sort_values(by=['scores'], ascending=[False])

,0,1,2,3,4,5,6,7,8,9,...,19995,19996,19997,19998,19999,qid,gene_id,value_1,y,scores
10187,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_013080,32.54460,3,5.600347
9510,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_012066,166.12200,3,5.448623
39,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_000050,17.75700,3,5.369223
1155,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_001315,21.04670,3,5.369223
1143,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_001302,16.56760,3,5.309357
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19719,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_026659,4.21160,2,-7.251520
13270,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_016928,206.08000,3,-7.251520
17620,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_022699,211.16300,3,-7.251520
18771,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_025379,7.45762,2,-7.251520


In [34]:
df_full['scores'].describe()

count    22154.000000
mean        -5.974493
std          1.340805
min         -7.251520
25%         -6.979959
50%         -6.638282
75%         -5.262270
max          5.600347
Name: scores, dtype: float64

In [35]:
df_full.sort_values(by=['value_1'], ascending=[False])

,0,1,2,3,4,5,6,7,8,9,...,19995,19996,19997,19998,19999,qid,gene_id,value_1,y,scores
4492,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_005626,12870.90,3,-6.979959
15660,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_020078,9933.06,3,-6.979959
10487,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_013407,8005.12,3,-0.654857
421,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_000470,5475.90,3,-6.979959
1859,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_002089,5094.75,3,-6.979959
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12549,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_015856,0.00,0,-4.029130
12546,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_015853,0.00,0,-6.979959
12538,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_015845,0.00,0,-6.979959
12519,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_015825,0.00,0,-6.979959


In [36]:
df_full[["gene_id", "value_1", "y", "scores"]].to_csv("HepG2_ranking_quartile.csv")

# Evaluation

In [ ]:
print(y)
print(scores)